# quant-kit — GGUF Quantization Pipeline
> **GitHub**: [DhruvalPtl/quant-kit](https://github.com/DhruvalPtl/quant-kit) | **HF**: [Dhptl](https://huggingface.co/Dhptl)

**Setup:** `Runtime → Change runtime type → T4 GPU` · Add `HF_TOKEN` to Colab Secrets (🔑)

| First time | Resume (morning) |
|---|---|
| Cells 1 → 2 → 3 → 4 → 5 → 6 | Cells 1 → 2 → 3 → 4 → 7 → 8 → 9 → 10 |

In [ ]:
# Cell 1 — Runtime check
import subprocess, shutil, psutil

gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader',
                     shell=True, capture_output=True, text=True)
print(f'[OK] GPU  : {gpu.stdout.strip()}' if gpu.returncode == 0
      else '[!!] No GPU — go to Runtime > Change runtime type > T4 GPU')
disk = shutil.disk_usage('/')
print(f'[OK] Disk : {disk.free/1e9:.1f} GB free of {disk.total/1e9:.1f} GB')
ram = psutil.virtual_memory()
print(f'[OK] RAM  : {ram.available/1e9:.1f} GB of {ram.total/1e9:.1f} GB')

In [ ]:
# Cell 2 — Clone & setup
# Set FORCE_REINSTALL = True only if Cell 3 reports llama-quantize errors
import os, shutil, subprocess as _sp
from pathlib import Path

FORCE_REINSTALL = False

WORKDIR = '/content/quant-kit'
if os.path.exists(WORKDIR):
    os.system(f'git -C {WORKDIR} pull')
else:
    os.system(f'git clone https://github.com/DhruvalPtl/quant-kit.git {WORKDIR}')
os.chdir(WORKDIR)

LLAMA_CPP = Path(WORKDIR) / 'llama.cpp'
if FORCE_REINSTALL and LLAMA_CPP.exists():
    shutil.rmtree(str(LLAMA_CPP))
    print('[OK] Deleted llama.cpp/ for reinstall')

qbin = LLAMA_CPP / 'llama-quantize'
if qbin.exists():
    _env = {**os.environ, 'LD_LIBRARY_PATH': str(LLAMA_CPP)}
    if _sp.run([str(qbin), '--help'], capture_output=True, env=_env).returncode != 0:
        print('[!!] llama-quantize broken — auto-removing for reinstall')
        shutil.rmtree(str(LLAMA_CPP))

!python setup_linux.py

In [ ]:
# Cell 3 — Verify llama-quantize + fix .so symlinks
import os, re, subprocess
from pathlib import Path
from collections import defaultdict

LLAMA_CPP = Path('/content/quant-kit/llama.cpp')

for f in LLAMA_CPP.glob('*.so'):
    if not f.is_symlink():
        v = LLAMA_CPP / (f.name + '.0')
        if not v.exists(): os.symlink(f.name, str(v))

by_base = defaultdict(list)
for f in LLAMA_CPP.glob('lib*.so.*'):
    m = re.match(r'^(lib.+\.so)\.(\d+)\.\d+\.\d+$', f.name)
    if m and not f.is_symlink():
        by_base[m.group(1)].append((int(m.group(2)), f.name))
for base, versions in by_base.items():
    versions.sort(reverse=True)
    so_m = LLAMA_CPP / f'{base}.{versions[0][0]}'
    if not so_m.exists(): os.symlink(versions[0][1], str(so_m))

env  = {**os.environ, 'LD_LIBRARY_PATH': str(LLAMA_CPP)}
qbin = LLAMA_CPP / 'llama-quantize'
if not qbin.exists():
    print('[ERR] llama-quantize not found — re-run Cell 2 with FORCE_REINSTALL = True')
else:
    r = subprocess.run([str(qbin), '--help'], capture_output=True, text=True, env=env)
    if r.returncode == 0:
        print(f'[OK] llama-quantize works  ({qbin.stat().st_size} bytes)')
        print('[OK] Ready to proceed')
    else:
        print(f'[ERR] llama-quantize failed (exit {r.returncode})')
        print(r.stderr[:300])
        missing = [l.strip() for l in
                   subprocess.run(['ldd', str(qbin)], capture_output=True,
                                  text=True, env=env).stdout.splitlines()
                   if 'not found' in l]
        if missing: print('Missing libs:', missing)
        print('Fix: re-run Cell 2 with FORCE_REINSTALL = True')

In [ ]:
# Cell 4 — HuggingFace login
import os
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    with open('/content/quant-kit/.env', 'w') as f:
        f.write(f'hf_token = "{hf_token}"\n')
    from huggingface_hub import HfApi
    print(f'[OK] Logged in as: {HfApi(token=hf_token).whoami()["name"]}')
except Exception as e:
    print(f'[ERR] {e}')
    print('Add HF_TOKEN to Colab Secrets (🔑) and enable notebook access')

In [ ]:
# Cell 5 — Quantize  ← FIRST TIME ONLY, skip on resume
import os
from pathlib import Path

MODEL_ID = 'google/gemma-4-12B'
QUANTS   = 'Q4_K_M Q5_K_M Q8_0 IQ4_XS'

os.chdir('/content/quant-kit')
MODEL_NAME = MODEL_ID.split('/')[-1]
out = Path('output') / MODEL_NAME

existing = list(out.glob('*.gguf')) if out.exists() else []
if existing:
    print('[OK] GGUFs already exist:')
    for f in sorted(existing): print(f'  {f.name}  ({f.stat().st_size/1e9:.1f} GB)')
else:
    !python quantize.py --model {MODEL_ID} --quants {QUANTS} --delete-src

In [ ]:
# Cell 6 — Save to Drive  ← run before sleep
from google.colab import drive
import shutil
from pathlib import Path

MODEL_NAME = 'gemma-4-12B'
drive.mount('/content/drive')

SRC  = Path(f'/content/quant-kit/output/{MODEL_NAME}')
DEST = Path(f'/content/drive/MyDrive/quant-kit-output/{MODEL_NAME}')
DEST.mkdir(parents=True, exist_ok=True)

for f in sorted(SRC.glob('*.gguf')):
    d = DEST / f.name
    if d.exists(): print(f'  [skip] {f.name}')
    else:
        print(f'  Copying {f.name} ({f.stat().st_size/1e9:.1f} GB)...')
        shutil.copy2(str(f), str(d))
        print(f'  [OK]')
print('\nSaved. Resume tomorrow: Cells 1→2→3→4→7→8→9→10')

In [ ]:
# Cell 7 — Restore from Drive  ← morning resume (instead of Cell 5)
from google.colab import drive
import shutil
from pathlib import Path

MODEL_NAME = 'gemma-4-12B'
drive.mount('/content/drive')

SRC  = Path(f'/content/drive/MyDrive/quant-kit-output/{MODEL_NAME}')
DEST = Path(f'/content/quant-kit/output/{MODEL_NAME}')
DEST.mkdir(parents=True, exist_ok=True)

files = sorted(SRC.glob('*.gguf')) if SRC.exists() else []
if not files:
    print(f'[ERR] No GGUFs in Drive at: {SRC}')
else:
    for f in files:
        d = DEST / f.name
        if d.exists(): print(f'  [skip] {f.name}')
        else:
            print(f'  Restoring {f.name} ({f.stat().st_size/1e9:.1f} GB)...')
            shutil.copy2(str(f), str(d))
            print(f'  [OK]')
    print('\n[OK] Run Cell 8 (benchmark)')

In [ ]:
# Cell 8 — Benchmark (uses llama-bench binary, shows live timer)
import os
os.chdir('/content/quant-kit')

MODEL_NAME = 'gemma-4-12B'

# ngl=99: try to offload all layers to GPU
# ngl=0:  CPU only
!python benchmark.py --model {MODEL_NAME} --ngl 99

In [ ]:
# Cell 9 — Generate model card
import os
os.chdir('/content/quant-kit')

MODEL_NAME = 'gemma-4-12B'
MODEL_ID   = 'google/gemma-4-12B'
HF_AUTHOR  = 'Dhptl'

!python model_card.py --model {MODEL_NAME} --original {MODEL_ID} --author {HF_AUTHOR}

In [ ]:
# Cell 10 — Upload to HuggingFace → creates Dhptl/gemma-4-12B-GGUF
import os
os.chdir('/content/quant-kit')

MODEL_NAME = 'gemma-4-12B'
HF_AUTHOR  = 'Dhptl'

!python upload.py --model {MODEL_NAME} --author {HF_AUTHOR}

In [ ]:
# Cell 11 — Cleanup local files (run after upload)
import shutil
from pathlib import Path

MODEL_NAME = 'gemma-4-12B'
output = Path(f'/content/quant-kit/output/{MODEL_NAME}')
if output.exists():
    shutil.rmtree(str(output))
    print(f'[OK] Cleaned: {output}')
disk = shutil.disk_usage('/')
print(f'[OK] Disk: {disk.free/1e9:.1f} GB free')